This notebook is a simple example of bounding an experimental design and simulating a few replicates.

# Imports

In [1]:
import sys
import os
import scMPRAforge as scm
import pandas as pd
import numpy as np
import dask.dataframe as dd


%load_ext autoreload
%autoreload 2

2025-09-22 18:33:33.747329: I tensorflow/core/util/util.cc:169] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-09-22 18:33:33.751441: W tensorflow/stream_executor/platform/default/dso_loader.cc:64] Could not load dynamic library 'libcudart.so.11.0'; dlerror: libcudart.so.11.0: cannot open shared object file: No such file or directory; LD_LIBRARY_PATH: /vast/palmer/apps/avx2/software/Code-Server/4.17.0/lib:/vast/palmer/apps/avx2/software/gettext/0.22.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libiconv/1.17-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/ncurses/6.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/libxml2/2.12.7-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/XZ/5.4.5-GCCcore-13.3.0/lib:/vast/palmer/apps/avx2/software/expat/2.6.2-GCCcore-13.3.0/lib:/vast/palmer/apps/av

# Create cluster

In [2]:
from dask.distributed import Client, LocalCluster
cluster=LocalCluster(memory_limit='8GB')
client=Client(cluster)

# Seting experimiental design parameters

In [3]:
#first, we define the new parameters we want to assign to this object.

new_cell_number=pd.Series({"reference":500,"blood":500})

#we make up 3x replicates
new_zi=pd.Series({"replicate_A":0.02,"replicate_B":0.021})

new_min=1
new_max=200

new_MOI=60

In [4]:
#next, let's create a bounds object from these parameters and the bounds of the shendure data.
artificial_bounds=scm.SHENDURE_BOUNDS.copy(
    min_mpra_umi=new_min,
    max_mpra_umi=new_max,
    zi=new_zi,
    cells_per_cell_type=new_cell_number)
    
artificial_bounds.set_effective_moi(new_MOI)

# Creating an artificial library

In [5]:
#making up the CREs
spread_gt,spread_hypothesis=scm.simple_spread(cell_types=new_cell_number.keys(),
                  min=new_min,
                  max=new_max,
                  fineness=2)

library=scm.simulate_library(CREs=spread_gt["cre_id"],
                 library_model=artificial_bounds.library_model)

In [ ]:
library

# Performing de-novo simulation

In [6]:
batch_plus=scm.de_novo_simulation(
                        simulation_replicates=3,
                        experiment_bounds=artificial_bounds,
                        ground_truth=spread_gt,
                        library=library)

In [7]:
batch_plus.gamut(client)

percent lost 0.0
percent lost 0.0
percent lost 0.0


# Save

In [ ]:
data_root="/gpfs/gibbs/pi/reilly/tabula_data"

In [ ]:
batch_plus.save(data_root,"simulated/tiny_de_novo")

In [ ]:
spread_hypothesis.to_tsv(f"{data_root}/simulated/tiny_de_novo_hypotheses.tsv")

In [ ]:
cluster.close()